### 0) Colab + Drive + Library

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, pathlib

PROJECT_ROOT = "/content/drive/MyDrive/nextgen_nlp_final"
DATA_DIR = f"{PROJECT_ROOT}/data"
ARTIFACTS_DIR = f"{PROJECT_ROOT}/artifacts"
FIG_DIR = f"{PROJECT_ROOT}/figures"
NB_DIR = f"{PROJECT_ROOT}/notebooks"
pathlib.Path(NB_DIR).mkdir(parents=True, exist_ok=True)

for d in [PROJECT_ROOT, DATA_DIR, ARTIFACTS_DIR, FIG_DIR]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: /content/drive/MyDrive/nextgen_nlp_final


In [3]:
!pip -q install pyarrow pandas numpy

In [4]:
import pandas as pd
import numpy as np

### 1）Read Data

In [5]:
df = pd.read_parquet(
    'https://storage.googleapis.com/msca-bdp-data-open/news_final_project/news_final_project.parquet',
    engine='pyarrow'
)

print("Shape:", df.shape)
df.head(3)

Shape: (199989, 5)


,url,date,language,title,text
0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod..."
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,\n\nThis AI video of gymnastics might be the f...
2,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,en,"If using AI feels like a chore, try this - Boi...","\n\nIf using AI feels like a chore, try this -..."


In [6]:
print(df.columns.tolist())

['url', 'date', 'language', 'title', 'text']


### 2）Data Profliling

missing

In [7]:
missing = (df.isna().mean() * 100).to_frame("missing_pct")
missing["missing_cnt"] = df.isna().sum()
missing["dtype"] = df.dtypes.astype(str)

missing

,missing_pct,missing_cnt,dtype
url,0.0,0,object
date,0.0,0,object
language,0.0,0,object
title,0.0,0,object
text,0.0,0,object


data col

In [8]:
URL_COL   = "url"
DATE_COL  = "date"
LANG_COL  = "language"
TITLE_COL = "title"
TEXT_COL  = "text"

In [9]:
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce", utc=True)

print("Parse success %:", df[DATE_COL].notna().mean() * 100)
print("Min date:", df[DATE_COL].min())
print("Max date:", df[DATE_COL].max())

Parse success %: 100.0
Min date: 2022-01-01 00:00:00+00:00
Max date: 2026-02-10 00:00:00+00:00


In [10]:
time_report = {
    "parse_success_pct": float(df[DATE_COL].notna().mean() * 100),
    "min_date_utc": str(df[DATE_COL].min()),
    "max_date_utc": str(df[DATE_COL].max()),
    "n_missing_date": int(df[DATE_COL].isna().sum())
}

import json
with open(f"{ARTIFACTS_DIR}/01_time_report.json", "w") as f:
    json.dump(time_report, f, indent=2)

 text length

In [11]:
text_len = df[TEXT_COL].fillna("").astype(str).str.len()

import numpy as np
percentiles = np.percentile(text_len, [0,25,50,75,90,95,99,100])

print("Text length percentiles:", percentiles)

Text length percentiles: [2.100000e+01 5.245000e+03 7.622000e+03 1.139000e+04 1.576600e+04
 1.941560e+04 3.383204e+04 5.807720e+05]


In [12]:
sample_df = df.sample(n=100, random_state=42)[
    [URL_COL, DATE_COL, TITLE_COL, TEXT_COL]
]

sample_df.to_json(
    f"{ARTIFACTS_DIR}/01_sample_100.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

In [13]:
print(np.percentile(text_len, [0,25,50,75,90,95,99,100]))

[2.100000e+01 5.245000e+03 7.622000e+03 1.139000e+04 1.576600e+04
 1.941560e+04 3.383204e+04 5.807720e+05]


### 3）Basic Filtering

In [14]:
import numpy as np
import pandas as pd

df0 = df.copy()

# English only
df1 = df0[df0[LANG_COL].astype(str).str.lower().eq("en")].copy()

# no super long/short text
text_len = df1[TEXT_COL].fillna("").astype(str).str.len()
MIN_LEN = 500
MAX_LEN = 50000

df1 = df1[(text_len >= MIN_LEN) & (text_len <= MAX_LEN)].copy()

print("After lang+len filter:", df1.shape)

After lang+len filter: (198010, 5)


### 4) Normalization

In [15]:
import re
import unicodedata

_whitespace_re = re.compile(r"\s+")
_null_re = re.compile(r"\x00")

def normalize_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)   # unicode 规范化
    s = _null_re.sub("", s)
    s = s.replace("\r", "\n")
    s = _whitespace_re.sub(" ", s).strip()
    return s

df1["title_norm"] = df1[TITLE_COL].map(normalize_text)
df1["text_norm"]  = df1[TEXT_COL].map(normalize_text)

print(df1[["title_norm","text_norm"]].head(2))

                                          title_norm  \
0  Bad Idea AI Price (BAD), Market Cap, Price Tod...   
1  This AI video of gymnastics might be the freak...   

                                           text_norm  
0  Bad Idea AI Price (BAD), Market Cap, Price Tod...  
1  This AI video of gymnastics might be the freak...  


In [16]:
df1.head()

,url,date,language,title,text,title_norm,text_norm
0,https://blockworks.co/price/bad,2025-06-23 00:00:00+00:00,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod..."
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01 00:00:00+00:00,en,This AI video of gymnastics might be the freak...,\n\nThis AI video of gymnastics might be the f...,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...
2,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22 00:00:00+00:00,en,"If using AI feels like a chore, try this - Boi...","\n\nIf using AI feels like a chore, try this -...","If using AI feels like a chore, try this - Boi...","If using AI feels like a chore, try this - Boi..."
3,https://citylife.capetown/gl/uncategorized/the...,2023-11-10 00:00:00+00:00,en,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation M...,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation Mode...
4,https://citylife.capetown/kk/uncategorized/mic...,2023-11-19 00:00:00+00:00,en,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers ...,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...


### 5) URL去重

In [17]:
import pandas as pd
import numpy as np
import gc

URL_COL="url"; DATE_COL="date"; LANG_COL="language"
TITLE_NORM_COL="title_norm"; TEXT_NORM_COL="text_norm"

# 0) 确保 date 是 datetime（你已经做过也没问题）
df1[DATE_COL] = pd.to_datetime(df1[DATE_COL], errors="coerce", utc=True)

# 1) 英文过滤（稳）
df_tmp = df1[df1[LANG_COL].astype(str).str.lower().eq("en")]

# 2) 长度过滤（用你之前的分布：大部分 < 33k，max 超大）
lens = df_tmp[TEXT_NORM_COL].fillna("").astype(str).str.len()
MIN_LEN = 500
MAX_LEN = 50000
df_tmp = df_tmp[(lens >= MIN_LEN) & (lens <= MAX_LEN)]
del lens
gc.collect()

# 3) URL 去重（非常建议做）
df_tmp = df_tmp.drop_duplicates(subset=[URL_COL])

# 4) 只保留你要的三列（date/title_norm/text_norm）
df_clean_v1 = df_tmp[[DATE_COL, TITLE_NORM_COL, TEXT_NORM_COL]].rename(
    columns={TITLE_NORM_COL:"title_clean", TEXT_NORM_COL:"text_clean"}
).copy()

print("Clean v1 shape:", df_clean_v1.shape)
df_clean_v1.head(3)

Clean v1 shape: (197985, 3)


,date,title_clean,text_clean
0,2025-06-23 00:00:00+00:00,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod..."
1,2024-07-01 00:00:00+00:00,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...
2,2024-09-22 00:00:00+00:00,"If using AI feels like a chore, try this - Boi...","If using AI feels like a chore, try this - Boi..."


### 保存到Drive

In [18]:
CLEAN_PATH = f"{ARTIFACTS_DIR}/clean_news_v1.parquet"
df_clean_v1.to_parquet(CLEAN_PATH, index=False)
print("Saved:", CLEAN_PATH)

Saved: /content/drive/MyDrive/nextgen_nlp_final/artifacts/clean_news_v1.parquet
